# 03 — RFM 客户价值分群

**目标**: 用 RFM 模型将客户分为不同价值层级，为精准营销提供依据。

**RFM 模型**:
- **R**ecency（最近一次购买距今多久）：越低越好
- **F**requency（购买频率）：越高越好
- **M**onetary（消费金额）：越高越好

**步骤**:
1. 计算每个客户的 R、F、M 值
2. 对 R、F、M 分别打分 (1-4 分)
3. 组合分数进行客户分群
4. 分析各群特征 & 给出营销建议

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 120
PALETTE = sns.color_palette('Set2', 8)

print('Libraries loaded.')

In [ ]:
# 加载数据
df = pd.read_csv('../data/Superstore_Cleaned.csv', parse_dates=['Order Date', 'Ship Date'])
print(f'Data: {len(df)} rows, {df["Customer ID"].nunique()} unique customers')

# 设定参考日期（数据中最后一天 + 1 天）
REFERENCE_DATE = df['Order Date'].max() + pd.Timedelta(days=1)
print(f'Reference Date: {REFERENCE_DATE.date()}')

In [ ]:
# ========== 1. 计算 RFM ==========
rfm = df.groupby('Customer ID').agg(
    Recency=('Order Date', lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency=('Order ID', 'nunique'),
    Monetary=('Sales', 'sum'),
    # 附加信息
    Avg_Basket=('Sales', 'mean'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean'),
    Segment=('Segment', 'first'),
    Region=('Region', 'first')
).reset_index()

print(f'RFM table: {len(rfm)} customers')
print(f'\nRFM 描述统计:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(1).to_string())

In [ ]:
# ========== 2. RFM 分布可视化 ==========
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R: 越低越好 → 左偏
ax = axes[0]
ax.hist(rfm['Recency'], bins=40, color=PALETTE[0], edgecolor='white', alpha=0.8)
ax.axvline(rfm['Recency'].median(), color='red', linestyle='--', label=f'Median: {rfm["Recency"].median():.0f}d')
ax.set_xlabel('Days Since Last Purchase')
ax.set_ylabel('Customer Count')
ax.set_title('R — Recency Distribution', fontweight='bold')
ax.legend()

# F: 越高越好
ax = axes[1]
ax.hist(rfm['Frequency'], bins=30, color=PALETTE[1], edgecolor='white', alpha=0.8)
ax.axvline(rfm['Frequency'].median(), color='red', linestyle='--', label=f'Median: {rfm["Frequency"].median():.0f}')
ax.set_xlabel('Order Count')
ax.set_ylabel('Customer Count')
ax.set_title('F — Frequency Distribution', fontweight='bold')
ax.legend()

# M: 越高越好
ax = axes[2]
ax.hist(rfm['Monetary']/1000, bins=40, color=PALETTE[2], edgecolor='white', alpha=0.8)
ax.axvline(rfm['Monetary'].median()/1000, color='red', linestyle='--', label=f'Median: ${rfm["Monetary"].median():,.0f}')
ax.set_xlabel('Total Spend ($K)')
ax.set_ylabel('Customer Count')
ax.set_title('M — Monetary Distribution', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 3. RFM 打分（四分位数法） ==========
# Recency: 值越小分越高（最近买的客户得分高）
r_labels = [4, 3, 2, 1]  # 4 = 最近买过 (最好), 1 = 很久没买 (最差)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=r_labels)

# Frequency: 值越大分越高
f_labels = [1, 2, 3, 4]  # 4 = 购买频繁 (最好), 1 = 很少买 (最差)
rfm['F_Score'] = pd.qcut(rfm['Frequency'], q=4, labels=f_labels, duplicates='drop')

# Monetary: 值越大分越高
m_labels = [1, 2, 3, 4]  # 4 = 花费多 (最好), 1 = 花费少 (最差)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=m_labels, duplicates='drop')

# 转换为整数
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# 综合得分
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print('RFM Scoring completed.')
print(f'Score range: {rfm["RFM_Score"].min()} ~ {rfm["RFM_Score"].max()}')
print(f'\nScore distribution:')
print(rfm['RFM_Score'].value_counts().sort_index())

In [ ]:
# ========== 4. 客户分群 ==========
def segment_customer(row):
    """
    基于 RFM 分数组合进行客户分群
    高分 = 4, 3; 低分 = 1, 2
    """
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champions'
    elif r >= 3 and f >= 3 and m < 3:
        return 'Loyal Customers'
    elif r >= 3 and f < 3 and m >= 3:
        return 'Big Spenders'
    elif r >= 3 and f < 3 and m < 3:
        return 'New/Promising'
    elif r < 3 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r < 3 and f >= 3 and m < 3:
        return 'Needs Attention'
    elif r < 3 and f < 3 and m >= 3:
        return 'Lost Big Spenders'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# 汇总
segment_summary = rfm.groupby('Segment').agg(
    Count=('Customer ID', 'nunique'),
    Pct=('Customer ID', lambda x: round(len(x)/len(rfm)*100, 1)),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum'),
    Total_Profit=('Total_Profit', 'sum'),
    Avg_Discount=('Avg_Discount', 'mean')
).sort_values('Avg_Monetary', ascending=False)

print('========== 客户分群汇总 ==========')
print(segment_summary.to_string())
print(f'\n客户总数: {len(rfm)}')

In [ ]:
# ========== 5. 可视化客户分群 ==========

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 柱状图：各群组人数
ax = axes[0, 0]
seg_order = segment_summary.sort_values('Count', ascending=False)
bars = ax.barh(range(len(seg_order)), seg_order['Count'], 
               color=[PALETTE[i % 8] for i in range(len(seg_order))])
ax.set_yticks(range(len(seg_order)))
ax.set_yticklabels(seg_order.index)
ax.invert_yaxis()
ax.set_xlabel('Customer Count')
ax.set_title('Customer Count by RFM Segment', fontweight='bold')
for i, (bar, pct) in enumerate(zip(bars, seg_order['Pct'])):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{pct}%', va='center')

# 气泡图：Avg Frequency × Avg Monetary
ax = axes[0, 1]
scatter = ax.scatter(
    segment_summary['Avg_Frequency'], 
    segment_summary['Avg_Monetary']/1000,
    s=segment_summary['Count']*2, alpha=0.6,
    c=range(len(segment_summary)), cmap='Set2'
)
for seg, row in segment_summary.iterrows():
    ax.annotate(seg, (row['Avg_Frequency'], row['Avg_Monetary']/1000),
                fontsize=9, ha='center', va='bottom')
ax.set_xlabel('Avg Frequency (Orders)')
ax.set_ylabel('Avg Monetary ($K)')
ax.set_title('Segment Map: Frequency × Monetary', fontweight='bold')

# 收入贡献
ax = axes[1, 0]
revenue_pct = segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100
wedges, texts, autotexts = ax.pie(
    revenue_pct.values, labels=revenue_pct.index, autopct='%1.1f%%',
    colors=sns.color_palette('Set2', len(segment_summary)), startangle=90
)
ax.set_title('Revenue Contribution by Segment', fontweight='bold')

# 利润贡献
ax = axes[1, 1]
profit_pct = segment_summary['Total_Profit'] / segment_summary['Total_Profit'].sum() * 100
bars = ax.bar(range(len(profit_pct)), profit_pct.values, 
              color=sns.color_palette('Set2', len(segment_summary)))
ax.set_xticks(range(len(profit_pct)))
ax.set_xticklabels(profit_pct.index, rotation=45, ha='right')
ax.set_ylabel('Profit Contribution (%)')
ax.set_title('Profit Contribution by Segment', fontweight='bold')
for bar, val in zip(bars, profit_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%',
            ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 6. 3D RFM 散点图 (Plotly 交互式) ==========
fig = px.scatter_3d(
    rfm, x='Recency', y='Frequency', z='Monetary',
    color='Segment', size='Monetary', opacity=0.7,
    title='3D RFM Customer Segmentation',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=700)
fig.show()

In [ ]:
# ========== 7. 客群 × 区域交叉分析 ==========
cross_tab = pd.crosstab(rfm['Segment'], rfm['Region'], values=rfm['Monetary'], 
                         aggfunc='sum', normalize='columns') * 100

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cross_tab.round(1), annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': '% of Region Revenue'})
ax.set_title('Segment Revenue Contribution by Region (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Customer Segment')
plt.tight_layout()
plt.savefig('../reports/rfm_region_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ========== 8. 保存 RFM 结果 ==========
rfm.to_csv('../data/Superstore_RFM.csv', index=False)
print('RFM results saved to: ../data/Superstore_RFM.csv')
print(f'Columns: {list(rfm.columns)}')

---
## 营销策略建议

| 客群 | 策略 | 预期效果 |
|------|------|---------|
| 🏆 **Champions** | VIP 会员、新品内测、推荐奖励 | 品牌忠诚度、口碑传播 |
| 💎 **Loyal Customers** | 交叉销售、升级推荐 | 提升客单价 |
| 💰 **Big Spenders** | 个性化推荐、专属折扣 | 提高购买频率 |
| 🌱 **New/Promising** | 欢迎邮件、首购优惠、入门引导 | 激活并养成购买习惯 |
| ⚠️ **At Risk** | 专属召回优惠、满意度调研 | 挽回流失、理解痛点 |
| 🔔 **Needs Attention** | 限时促销、新品通知 | 重新激活 |
| 💸 **Lost Big Spenders** | 大力度召回：电话/定向优惠券 | 挽回高价值客户 |
| ❌ **Lost** | 低成本触达（如 EDM）、观察 | 若 ROI 低则放弃 |

---

## 项目总结

恭喜！你完成了完整的 Superstore 数据分析项目：

| 阶段 | 产出 |
|------|------|
| 数据清洗 | `Superstore_Cleaned.csv` — 干净的分析数据集 |
| 探索性分析 | 6 个维度的可视化 & 业务洞察 |
| RFM 建模 | 8 个客户群组的营销策略 |
| Tableau 仪表板 | 交互式销售看板 |

**下一步**:
- 将数据导入 Tableau → 搭建交互式仪表板
- 将项目推送到 GitHub
- 在简历中添加本项目